# Standardize Metadata

Standardizes Bronze SQL Server metadata into canonical Work metadata structures. The notebook selects the attributes required by the canonical model, normalizes source values, generates deterministic identifiers, preserves source lineage, and prepares candidate metadata for validation before promotion to Silver.

## Input Parameters

Defines the source database to process and the execution step responsible for the current standardization process.

In [1]:
p_source_database = "DataOps_Control"
p_source_execution_step_ids = "{}"
p_metadata_entities = "[]"

StatementMeta(, a683c6b6-3001-4232-958d-e4993d1e7ad9, 3, Finished, Available, Finished, False)

## Imports

Imports the utilities required for metadata transformation and notebook output.

In [2]:
import json

from pyspark.sql import functions as F

StatementMeta(, a683c6b6-3001-4232-958d-e4993d1e7ad9, 4, Finished, Available, Finished, False)

## Helper Functions

In [3]:
def get_metadata_objects_lookup(
    source_database,
    source_execution_step_id
):
    """
    Returns the latest Bronze metadata object snapshot required for
    canonical object ID resolution during downstream metadata processing.
    """
    return (
        spark.table("bronze.metadata_objects")
        .filter(
            (F.col("source_database") == source_database) &
            (F.col("execution_step_id") == source_execution_step_id)
        )
        .select(
            F.trim(F.col("source_database")).alias("source_database"),
            F.trim(F.col("source_schema")).alias("source_schema"),
            F.trim(F.col("object_name")).alias("object_name")
        )
        .withColumn(
            "object_id",
            F.sha2(
                F.concat_ws(
                    "||",
                    F.coalesce(F.col("source_database"), F.lit("<NULL>")),
                    F.coalesce(F.col("source_schema"), F.lit("<NULL>")),
                    F.coalesce(F.col("object_name"), F.lit("<NULL>"))
                ),
                256
            )
        )
        .select(
            "object_id",
            "source_database",
            "source_schema",
            "object_name"
        )
    )

StatementMeta(, a683c6b6-3001-4232-958d-e4993d1e7ad9, 5, Finished, Available, Finished, False)

In [4]:
def get_metadata_constraints_lookup(
    source_database,
    source_execution_step_id,
    objects_source_execution_step_id
):
    """
    Returns the latest Bronze constraint snapshot with canonical constraint IDs.
    """

    constraints = (
        spark.table("bronze.metadata_constraints")
        .filter(
            (F.col("source_database") == source_database) &
            (F.col("execution_step_id") == source_execution_step_id)
        )
        .select(
            "source_database",
            "source_schema",
            "object_name",
            "constraint_name"
        )
        .withColumn("source_database", F.trim(F.col("source_database")))
        .withColumn("source_schema", F.trim(F.col("source_schema")))
        .withColumn("object_name", F.trim(F.col("object_name")))
        .withColumn("constraint_name", F.trim(F.col("constraint_name")))
    )

    objects_lookup = get_metadata_objects_lookup(
        source_database=source_database,
        source_execution_step_id=objects_source_execution_step_id
    )

    return (
        constraints.alias("c")
        .join(
            F.broadcast(objects_lookup).alias("o"),
            (
                (F.col("c.source_database") == F.col("o.source_database")) &
                (F.col("c.source_schema") == F.col("o.source_schema")) &
                (F.col("c.object_name") == F.col("o.object_name"))
            ),
            "left"
        )
        .withColumn(
            "constraint_id",
            F.sha2(
                F.concat_ws(
                    "||",
                    F.coalesce(F.col("o.object_id"), F.lit("<NULL>")),
                    F.coalesce(F.col("c.constraint_name"), F.lit("<NULL>"))
                ),
                256
            )
        )
        .select(
            "constraint_id",
            F.col("c.source_database"),
            F.col("c.source_schema"),
            F.col("c.object_name"),
            F.col("c.constraint_name")
        )
    )

StatementMeta(, a683c6b6-3001-4232-958d-e4993d1e7ad9, 6, Finished, Available, Finished, False)

In [5]:
def get_metadata_columns_lookup(
    source_database,
    source_execution_step_id,
    objects_source_execution_step_id
):
    """
    Returns the latest Bronze column snapshot with canonical column IDs.
    """

    columns = (
        spark.table("bronze.metadata_columns")
        .filter(
            (F.col("source_database") == source_database) &
            (F.col("execution_step_id") == source_execution_step_id)
        )
        .select(
            "source_database",
            "source_schema",
            "object_name",
            "column_name"
        )
        .withColumn("source_database", F.trim(F.col("source_database")))
        .withColumn("source_schema", F.trim(F.col("source_schema")))
        .withColumn("object_name", F.trim(F.col("object_name")))
        .withColumn("column_name", F.trim(F.col("column_name")))
    )

    objects_lookup = get_metadata_objects_lookup(
        source_database=source_database,
        source_execution_step_id=objects_source_execution_step_id
    )

    return (
        columns.alias("c")
        .join(
            F.broadcast(objects_lookup).alias("o"),
            (
                (F.col("c.source_database") == F.col("o.source_database")) &
                (F.col("c.source_schema") == F.col("o.source_schema")) &
                (F.col("c.object_name") == F.col("o.object_name"))
            ),
            "left"
        )
        .withColumn(
            "column_id",
            F.sha2(
                F.concat_ws(
                    "||",
                    F.coalesce(F.col("o.object_id"), F.lit("<NULL>")),
                    F.coalesce(F.col("c.column_name"), F.lit("<NULL>"))
                ),
                256
            )
        )
        .select(
            "column_id",
            F.col("c.source_database"),
            F.col("c.source_schema"),
            F.col("c.object_name"),
            F.col("c.column_name")
        )
    )

StatementMeta(, a683c6b6-3001-4232-958d-e4993d1e7ad9, 7, Finished, Available, Finished, False)

## Process Metadata Objects

Reads the latest Bronze snapshot for `metadata_objects`, standardizes object attributes, generates the canonical `object_id`, adds execution lineage, and replaces the current Work snapshot for the selected source database.

The function returns the standardized DataFrame so it can be reused by downstream metadata processing in the same notebook execution.

In [3]:
def process_metadata_objects(
    source_database,
    source_execution_step_id
):
    # Read latest Bronze snapshot
    bronze_metadata_objects = (
        spark.table("bronze.metadata_objects")
        .filter(
            (F.col("source_database") == source_database) &
            (F.col("execution_step_id") == source_execution_step_id)
        )
        .select(
            "source_database",
            "source_schema",
            "object_name",
            "object_type",
            "source_object_id",
            "source_modified_at",
            F.col("execution_step_id").alias("source_execution_step_id")
        )
    )

    # Normalize source attributes
    work_metadata_objects = (
        bronze_metadata_objects
        .withColumn("source_database", F.trim(F.col("source_database")))
        .withColumn("source_schema", F.trim(F.col("source_schema")))
        .withColumn("object_name", F.trim(F.col("object_name")))
        .withColumn(
            "object_type",
            F.when(F.col("object_type") == "USER_TABLE", "TABLE")
            .when(F.col("object_type") == "VIEW", "VIEW")
            .when(F.col("object_type") == "SQL_STORED_PROCEDURE", "PROCEDURE")
            .when(
                F.col("object_type").isin(
                    "SQL_SCALAR_FUNCTION",
                    "SQL_INLINE_TABLE_VALUED_FUNCTION",
                    "SQL_TABLE_VALUED_FUNCTION"
                ),
                "FUNCTION"
            )
            .otherwise(F.col("object_type"))
        )
    )

    # Generate deterministic canonical ID
    work_metadata_objects = (
        work_metadata_objects
        .withColumn(
            "object_id",
            F.sha2(
                F.concat_ws(
                    "||",
                    F.coalesce(F.col("source_database"), F.lit("<NULL>")),
                    F.coalesce(F.col("source_schema"), F.lit("<NULL>")),
                    F.coalesce(F.col("object_name"), F.lit("<NULL>"))
                ),
                256
            )
        )
    )

    # Select Work schema
    work_metadata_objects = (
        work_metadata_objects
        .select(
            "object_id",
            "source_database",
            "source_schema",
            "object_name",
            "object_type",
            "source_object_id",
            "source_modified_at",
            "source_execution_step_id"
        )
    )

    # Replace current Work snapshot for this database
    (
        work_metadata_objects.write
        .format("delta")
        .mode("overwrite")
        .option(
            "replaceWhere",
            f"source_database = '{source_database}'"
        )
        .saveAsTable("work.metadata_objects")
    )

    return work_metadata_objects

StatementMeta(, b30b3a7b-580a-483c-b4c1-23d22a1b1d61, 5, Finished, Available, Finished, False)

## Process Metadata Columns

Reads the latest Bronze snapshot for `metadata_columns`, standardizes column attributes, resolves the canonical parent `object_id`, generates the deterministic `column_id`, adds execution lineage, and replaces the current Work snapshot for the selected source database.

Object references are resolved from the latest valid Bronze metadata object snapshot identified by `DataOps_Control`.

In [7]:
def process_metadata_columns(
    source_database,
    source_execution_step_id,
    objects_source_execution_step_id
):
    # Read latest Bronze column snapshot
    bronze_metadata_columns = (
        spark.table("bronze.metadata_columns")
        .filter(
            (F.col("source_database") == source_database) &
            (F.col("execution_step_id") == source_execution_step_id)
        )
        .select(
            "source_database",
            "source_schema",
            "object_name",
            "column_name",
            "ordinal_position",
            "data_type",
            "max_length",
            "precision",
            "scale",
            "is_nullable",
            "is_identity",
            "is_computed",
            "default_definition",
            "computed_definition",
            F.col("execution_step_id").alias("source_execution_step_id")
        )
    )

    # Normalize attributes
    work_metadata_columns = (
        bronze_metadata_columns
        .withColumn("source_database", F.trim(F.col("source_database")))
        .withColumn("source_schema", F.trim(F.col("source_schema")))
        .withColumn("object_name", F.trim(F.col("object_name")))
        .withColumn("column_name", F.trim(F.col("column_name")))
        .withColumn(
            "data_type",
            F.lower(F.trim(F.col("data_type")))
        )
        .withColumn(
            "max_length",
            F.when(
                F.col("data_type").isin("nchar", "nvarchar") &
                (F.col("max_length") != -1),
                F.col("max_length") / 2
            )
            .otherwise(F.col("max_length"))
            .cast("int")
        )
        .withColumn(
            "default_definition",
            F.when(
                F.trim(F.col("default_definition")) == "",
                F.lit(None)
            ).otherwise(F.col("default_definition"))
        )
        .withColumn(
            "computed_definition",
            F.when(
                F.trim(F.col("computed_definition")) == "",
                F.lit(None)
            ).otherwise(F.col("computed_definition"))
        )
    )

    # Resolve canonical object ID
    objects_lookup = get_metadata_objects_lookup(
        source_database=source_database,
        source_execution_step_id=objects_source_execution_step_id
    )

    work_metadata_columns = (
        work_metadata_columns.alias("c")
        .join(
            F.broadcast(objects_lookup).alias("o"),
            (
                (F.col("c.source_database") == F.col("o.source_database")) &
                (F.col("c.source_schema") == F.col("o.source_schema")) &
                (F.col("c.object_name") == F.col("o.object_name"))
            ),
            "left"
        )
        .select(
            F.col("o.object_id"),
            F.col("c.*")
        )
    )

    # Generate deterministic canonical column ID
    work_metadata_columns = (
        work_metadata_columns
        .withColumn(
            "column_id",
            F.sha2(
                F.concat_ws(
                    "||",
                    F.coalesce(F.col("object_id"), F.lit("<NULL>")),
                    F.coalesce(F.col("column_name"), F.lit("<NULL>"))
                ),
                256
            )
        )
    )

    # Select Work schema
    work_metadata_columns = (
        work_metadata_columns
        .select(
            "column_id",
            "object_id",
            "source_database",
            "source_schema",
            "object_name",
            "column_name",
            "ordinal_position",
            "data_type",
            "max_length",
            "precision",
            "scale",
            "is_nullable",
            "is_identity",
            "is_computed",
            "default_definition",
            "computed_definition",
            "source_execution_step_id"
        )
    )

    # Replace current Work snapshot for this database
    (
        work_metadata_columns.write
        .format("delta")
        .mode("overwrite")
        .option(
            "replaceWhere",
            f"source_database = '{source_database}'"
        )
        .saveAsTable("work.metadata_columns")
    )

    return work_metadata_columns

StatementMeta(, b30b3a7b-580a-483c-b4c1-23d22a1b1d61, 9, Finished, Available, Finished, False)

## Process Metadata Constraints

Reads the latest Bronze snapshot for `metadata_constraints`, standardizes constraint attributes, resolves canonical IDs for the owning and referenced objects, generates the deterministic `constraint_id`, adds execution lineage, and replaces the current Work snapshot for the selected source database.

Object references are resolved from the latest valid Bronze metadata object snapshot identified by `DataOps_Control`.

In [9]:
def process_metadata_constraints(
    source_database,
    source_execution_step_id,
    objects_source_execution_step_id
):
    # Read latest Bronze constraint snapshot
    bronze_metadata_constraints = (
        spark.table("bronze.metadata_constraints")
        .filter(
            (F.col("source_database") == source_database) &
            (F.col("execution_step_id") == source_execution_step_id)
        )
        .select(
            "source_database",
            "source_schema",
            "object_name",
            "constraint_name",
            "constraint_type",
            "referenced_database",
            "referenced_schema",
            "referenced_object",
            "delete_action",
            "update_action",
            F.col("execution_step_id").alias("source_execution_step_id")
        )
    )

    # Normalize attributes
    work_metadata_constraints = (
        bronze_metadata_constraints
        .withColumn(
            "source_database",
            F.trim(F.col("source_database"))
        )
        .withColumn(
            "source_schema",
            F.trim(F.col("source_schema"))
        )
        .withColumn(
            "object_name",
            F.trim(F.col("object_name"))
        )
        .withColumn(
            "constraint_name",
            F.trim(F.col("constraint_name"))
        )
        .withColumn(
            "constraint_type",
            F.upper(F.trim(F.col("constraint_type")))
        )
        .withColumn(
            "referenced_database",
            F.when(
                F.trim(F.col("referenced_database")) == "",
                F.lit(None)
            ).otherwise(F.trim(F.col("referenced_database")))
        )
        .withColumn(
            "referenced_schema",
            F.when(
                F.trim(F.col("referenced_schema")) == "",
                F.lit(None)
            ).otherwise(F.trim(F.col("referenced_schema")))
        )
        .withColumn(
            "referenced_object",
            F.when(
                F.trim(F.col("referenced_object")) == "",
                F.lit(None)
            ).otherwise(F.trim(F.col("referenced_object")))
        )
        .withColumn(
            "delete_action",
            F.upper(F.trim(F.col("delete_action")))
        )
        .withColumn(
            "update_action",
            F.upper(F.trim(F.col("update_action")))
        )
    )

    # Load canonical object lookup from latest valid Bronze snapshot
    objects_lookup = get_metadata_objects_lookup(
        source_database=source_database,
        source_execution_step_id=objects_source_execution_step_id
    )

    # Resolve owning object ID
    work_metadata_constraints = (
        work_metadata_constraints.alias("c")
        .join(
            F.broadcast(objects_lookup).alias("o"),
            (
                (F.col("c.source_database") == F.col("o.source_database")) &
                (F.col("c.source_schema") == F.col("o.source_schema")) &
                (F.col("c.object_name") == F.col("o.object_name"))
            ),
            "left"
        )
        .select(
            F.col("o.object_id"),
            F.col("c.*")
        )
    )

    # Generate deterministic canonical constraint ID
    work_metadata_constraints = (
        work_metadata_constraints
        .withColumn(
            "constraint_id",
            F.sha2(
                F.concat_ws(
                    "||",
                    F.coalesce(F.col("object_id"), F.lit("<NULL>")),
                    F.coalesce(F.col("constraint_name"), F.lit("<NULL>"))
                ),
                256
            )
        )
    )

    # Resolve referenced object ID
    referenced_objects_lookup = (
        objects_lookup
        .select(
            F.col("object_id").alias("referenced_object_id"),
            F.col("source_database").alias("ref_database"),
            F.col("source_schema").alias("ref_schema"),
            F.col("object_name").alias("ref_object")
        )
    )

    work_metadata_constraints = (
        work_metadata_constraints.alias("c")
        .join(
            F.broadcast(referenced_objects_lookup).alias("r"),
            (
                (F.col("c.referenced_database") == F.col("r.ref_database")) &
                (F.col("c.referenced_schema") == F.col("r.ref_schema")) &
                (F.col("c.referenced_object") == F.col("r.ref_object"))
            ),
            "left"
        )
        .select(
            F.col("c.*"),
            F.col("r.referenced_object_id")
        )
    )

    # Select Work schema
    work_metadata_constraints = (
        work_metadata_constraints
        .select(
            "constraint_id",
            "object_id",
            "source_database",
            "source_schema",
            "object_name",
            "constraint_name",
            "constraint_type",
            "referenced_database",
            "referenced_schema",
            "referenced_object",
            "referenced_object_id",
            "delete_action",
            "update_action",
            "source_execution_step_id"
        )
    )

    # Replace current Work snapshot for this database
    (
        work_metadata_constraints.write
        .format("delta")
        .mode("overwrite")
        .option(
            "replaceWhere",
            f"source_database = '{source_database}'"
        )
        .saveAsTable("work.metadata_constraints")
    )

    return work_metadata_constraints

StatementMeta(, b30b3a7b-580a-483c-b4c1-23d22a1b1d61, 11, Finished, Available, Finished, False)

## Process Metadata Constraint Columns

Reads the latest Bronze snapshot for `metadata_constraint_columns`, standardizes constraint-column attributes, resolves the canonical constraint and column identifiers, generates the deterministic `constraint_column_id`, adds execution lineage, and replaces the current Work snapshot for the selected source database.

Constraint and column references are resolved from the latest valid Bronze metadata snapshots identified by `DataOps_Control`.

In [6]:
def process_metadata_constraint_columns(
    source_database,
    source_execution_step_id,
    constraints_source_execution_step_id,
    columns_source_execution_step_id,
    objects_source_execution_step_id
):
    # Read latest Bronze constraint-column snapshot
    bronze_metadata_constraint_columns = (
        spark.table("bronze.metadata_constraint_columns")
        .filter(
            (F.col("source_database") == source_database) &
            (F.col("execution_step_id") == source_execution_step_id)
        )
        .select(
            "source_database",
            "source_schema",
            "object_name",
            "constraint_name",
            "column_name",
            "referenced_column_name",
            "ordinal_position",
            F.col("execution_step_id").alias("source_execution_step_id")
        )
    )

    # Normalize attributes
    work_metadata_constraint_columns = (
        bronze_metadata_constraint_columns
        .withColumn("source_database", F.trim(F.col("source_database")))
        .withColumn("source_schema", F.trim(F.col("source_schema")))
        .withColumn("object_name", F.trim(F.col("object_name")))
        .withColumn("constraint_name", F.trim(F.col("constraint_name")))
        .withColumn("column_name", F.trim(F.col("column_name")))
        .withColumn(
            "referenced_column_name",
            F.when(
                F.trim(F.col("referenced_column_name")) == "",
                F.lit(None)
            ).otherwise(F.trim(F.col("referenced_column_name")))
        )
    )

    # Resolve canonical constraint ID
    constraints_lookup = get_metadata_constraints_lookup(
        source_database=source_database,
        source_execution_step_id=constraints_source_execution_step_id,
        objects_source_execution_step_id=objects_source_execution_step_id
    )

    work_metadata_constraint_columns = (
        work_metadata_constraint_columns.alias("cc")
        .join(
            F.broadcast(constraints_lookup).alias("c"),
            (
                (F.col("cc.source_database") == F.col("c.source_database")) &
                (F.col("cc.source_schema") == F.col("c.source_schema")) &
                (F.col("cc.object_name") == F.col("c.object_name")) &
                (F.col("cc.constraint_name") == F.col("c.constraint_name"))
            ),
            "left"
        )
        .select(
            F.col("c.constraint_id"),
            F.col("cc.*")
        )
    )

    # Resolve canonical column ID
    columns_lookup = get_metadata_columns_lookup(
        source_database=source_database,
        source_execution_step_id=columns_source_execution_step_id,
        objects_source_execution_step_id=objects_source_execution_step_id
    )

    work_metadata_constraint_columns = (
        work_metadata_constraint_columns.alias("cc")
        .join(
            F.broadcast(columns_lookup).alias("c"),
            (
                (F.col("cc.source_database") == F.col("c.source_database")) &
                (F.col("cc.source_schema") == F.col("c.source_schema")) &
                (F.col("cc.object_name") == F.col("c.object_name")) &
                (F.col("cc.column_name") == F.col("c.column_name"))
            ),
            "left"
        )
        .select(
            F.col("c.column_id"),
            F.col("cc.*")
        )
    )

    # Generate deterministic canonical constraint-column ID
    work_metadata_constraint_columns = (
        work_metadata_constraint_columns
        .withColumn(
            "constraint_column_id",
            F.sha2(
                F.concat_ws(
                    "||",
                    F.coalesce(F.col("constraint_id"), F.lit("<NULL>")),
                    F.coalesce(F.col("column_id"), F.lit("<NULL>")),
                    F.coalesce(
                        F.col("ordinal_position").cast("string"),
                        F.lit("<NULL>")
                    )
                ),
                256
            )
        )
    )

    # Select Work schema
    work_metadata_constraint_columns = (
        work_metadata_constraint_columns
        .select(
            "constraint_column_id",
            "constraint_id",
            "column_id",
            "source_database",
            "source_schema",
            "object_name",
            "constraint_name",
            "column_name",
            "referenced_column_name",
            "ordinal_position",
            "source_execution_step_id"
        )
    )

    # Replace current Work snapshot for this database
    (
        work_metadata_constraint_columns.write
        .format("delta")
        .mode("overwrite")
        .option(
            "replaceWhere",
            f"source_database = '{source_database}'"
        )
        .saveAsTable("work.metadata_constraint_columns")
    )

    return work_metadata_constraint_columns

StatementMeta(, a1afb0ff-abf2-45e1-8962-3e4c4e2bd619, 8, Finished, Available, Finished, False)

## Process Metadata Relationships

Reads the latest Bronze snapshot for `metadata_relationships`, standardizes relationship attributes, resolves canonical identifiers for the source and target objects and columns, generates the deterministic `relationship_id`, adds execution lineage, and replaces the current Work snapshot for the selected source database.

Object and column references are resolved from the latest valid Bronze metadata snapshots identified by `DataOps_Control`.

In [6]:
def process_metadata_relationships(
    source_database,
    source_execution_step_id,
    objects_source_execution_step_id,
    columns_source_execution_step_id
):
    # Read latest Bronze relationship snapshot
    bronze_metadata_relationships = (
        spark.table("bronze.metadata_relationships")
        .filter(
            (F.col("source_database") == source_database) &
            (F.col("execution_step_id") == source_execution_step_id)
        )
        .select(
            "source_database",
            "source_schema",
            "source_object",
            "source_column",
            "target_database",
            "target_schema",
            "target_object",
            "target_column",
            "relationship_type",
            "relationship_name",
            F.col("execution_step_id").alias("source_execution_step_id")
        )
    )

    # Normalize attributes
    work_metadata_relationships = (
        bronze_metadata_relationships
        .withColumn("source_database", F.trim(F.col("source_database")))
        .withColumn("source_schema", F.trim(F.col("source_schema")))
        .withColumn("source_object", F.trim(F.col("source_object")))
        .withColumn("source_column", F.trim(F.col("source_column")))
        .withColumn("target_database", F.trim(F.col("target_database")))
        .withColumn("target_schema", F.trim(F.col("target_schema")))
        .withColumn("target_object", F.trim(F.col("target_object")))
        .withColumn("target_column", F.trim(F.col("target_column")))
        .withColumn(
            "relationship_type",
            F.upper(F.trim(F.col("relationship_type")))
        )
        .withColumn(
            "relationship_name",
            F.when(
                F.trim(F.col("relationship_name")) == "",
                F.lit(None)
            ).otherwise(F.trim(F.col("relationship_name")))
        )
    )

    # Load canonical object lookup
    objects_lookup = get_metadata_objects_lookup(
        source_database=source_database,
        source_execution_step_id=objects_source_execution_step_id
    )

    # Resolve source object ID
    work_metadata_relationships = (
        work_metadata_relationships.alias("r")
        .join(
            F.broadcast(objects_lookup).alias("so"),
            (
                (F.col("r.source_database") == F.col("so.source_database")) &
                (F.col("r.source_schema") == F.col("so.source_schema")) &
                (F.col("r.source_object") == F.col("so.object_name"))
            ),
            "left"
        )
        .select(
            F.col("so.object_id").alias("source_object_id"),
            F.col("r.*")
        )
    )

    # Resolve target object ID
    work_metadata_relationships = (
        work_metadata_relationships.alias("r")
        .join(
            F.broadcast(objects_lookup).alias("to"),
            (
                (F.col("r.target_database") == F.col("to.source_database")) &
                (F.col("r.target_schema") == F.col("to.source_schema")) &
                (F.col("r.target_object") == F.col("to.object_name"))
            ),
            "left"
        )
        .select(
            F.col("r.*"),
            F.col("to.object_id").alias("target_object_id")
        )
    )

    # Load canonical column lookup
    columns_lookup = get_metadata_columns_lookup(
        source_database=source_database,
        source_execution_step_id=columns_source_execution_step_id,
        objects_source_execution_step_id=objects_source_execution_step_id
    )

    # Resolve source column ID
    work_metadata_relationships = (
        work_metadata_relationships.alias("r")
        .join(
            F.broadcast(columns_lookup).alias("sc"),
            (
                (F.col("r.source_database") == F.col("sc.source_database")) &
                (F.col("r.source_schema") == F.col("sc.source_schema")) &
                (F.col("r.source_object") == F.col("sc.object_name")) &
                (F.col("r.source_column") == F.col("sc.column_name"))
            ),
            "left"
        )
        .select(
            F.col("r.*"),
            F.col("sc.column_id").alias("source_column_id")
        )
    )

    # Resolve target column ID
    work_metadata_relationships = (
        work_metadata_relationships.alias("r")
        .join(
            F.broadcast(columns_lookup).alias("tc"),
            (
                (F.col("r.target_database") == F.col("tc.source_database")) &
                (F.col("r.target_schema") == F.col("tc.source_schema")) &
                (F.col("r.target_object") == F.col("tc.object_name")) &
                (F.col("r.target_column") == F.col("tc.column_name"))
            ),
            "left"
        )
        .select(
            F.col("r.*"),
            F.col("tc.column_id").alias("target_column_id")
        )
    )

    # Generate deterministic canonical relationship ID
    work_metadata_relationships = (
        work_metadata_relationships
        .withColumn(
            "relationship_id",
            F.sha2(
                F.concat_ws(
                    "||",
                    F.coalesce(F.col("source_object_id"), F.lit("<NULL>")),
                    F.coalesce(F.col("source_column_id"), F.lit("<NULL>")),
                    F.coalesce(F.col("target_object_id"), F.lit("<NULL>")),
                    F.coalesce(F.col("target_column_id"), F.lit("<NULL>")),
                    F.coalesce(F.col("relationship_type"), F.lit("<NULL>"))
                ),
                256
            )
        )
    )

    # Select Work schema
    work_metadata_relationships = (
        work_metadata_relationships
        .select(
            "relationship_id",
            "source_database",
            "source_schema",
            "source_object",
            "source_column",
            "target_database",
            "target_schema",
            "target_object",
            "target_column",
            "relationship_type",
            "relationship_name",
            "source_execution_step_id"
        )
    )

    # Replace current Work snapshot for this database
    (
        work_metadata_relationships.write
        .format("delta")
        .mode("overwrite")
        .option(
            "replaceWhere",
            f"source_database = '{source_database}'"
        )
        .saveAsTable("work.metadata_relationships")
    )

    return work_metadata_relationships

StatementMeta(, a683c6b6-3001-4232-958d-e4993d1e7ad9, 8, Finished, Available, Finished, False)

## Process Metadata Modules

Reads the latest Bronze snapshot for `metadata_modules`, standardizes module attributes and object types, resolves the canonical parent `object_id`, generates the module identifier, adds execution lineage, and replaces the current Work snapshot for the selected source database.

Object references are resolved from the latest valid Bronze metadata object snapshot identified by `DataOps_Control`.

In [8]:
def process_metadata_modules(
    source_database,
    source_execution_step_id,
    objects_source_execution_step_id
):
    # Read latest Bronze module snapshot
    bronze_metadata_modules = (
        spark.table("bronze.metadata_modules")
        .filter(
            (F.col("source_database") == source_database) &
            (F.col("execution_step_id") == source_execution_step_id)
        )
        .select(
            "source_database",
            "source_schema",
            "object_name",
            "object_type",
            "module_definition",
            F.col("execution_step_id").alias("source_execution_step_id")
        )
    )

    # Normalize attributes
    work_metadata_modules = (
        bronze_metadata_modules
        .withColumn(
            "source_database",
            F.trim(F.col("source_database"))
        )
        .withColumn(
            "source_schema",
            F.trim(F.col("source_schema"))
        )
        .withColumn(
            "object_name",
            F.trim(F.col("object_name"))
        )
        .withColumn(
            "object_type",
            F.when(
                F.col("object_type") == "VIEW",
                "VIEW"
            )
            .when(
                F.col("object_type") == "SQL_STORED_PROCEDURE",
                "PROCEDURE"
            )
            .when(
                F.col("object_type").isin(
                    "SQL_SCALAR_FUNCTION",
                    "SQL_INLINE_TABLE_VALUED_FUNCTION",
                    "SQL_TABLE_VALUED_FUNCTION"
                ),
                "FUNCTION"
            )
            .otherwise(
                F.upper(F.trim(F.col("object_type")))
            )
        )
        .withColumn(
            "module_definition",
            F.when(
                F.trim(F.col("module_definition")) == "",
                F.lit(None)
            ).otherwise(F.col("module_definition"))
        )
    )

    # Load canonical object lookup
    objects_lookup = get_metadata_objects_lookup(
        source_database=source_database,
        source_execution_step_id=objects_source_execution_step_id
    )

    # Resolve canonical object ID
    work_metadata_modules = (
        work_metadata_modules.alias("m")
        .join(
            F.broadcast(objects_lookup).alias("o"),
            (
                (F.col("m.source_database") == F.col("o.source_database")) &
                (F.col("m.source_schema") == F.col("o.source_schema")) &
                (F.col("m.object_name") == F.col("o.object_name"))
            ),
            "left"
        )
        .select(
            F.col("o.object_id"),
            F.col("m.*")
        )
    )

    # Generate deterministic module ID
    work_metadata_modules = (
        work_metadata_modules
        .withColumn(
            "module_id",
            F.sha2(
                F.coalesce(
                    F.col("object_id"),
                    F.lit("<NULL>")
                ),
                256
            )
        )
    )

    # Select Work schema
    work_metadata_modules = (
        work_metadata_modules
        .select(
            "module_id",
            "object_id",
            "source_database",
            "source_schema",
            "object_name",
            "object_type",
            "module_definition",
            "source_execution_step_id"
        )
    )

    # Replace current Work snapshot for this database
    (
        work_metadata_modules.write
        .format("delta")
        .mode("overwrite")
        .option(
            "replaceWhere",
            f"source_database = '{source_database}'"
        )
        .saveAsTable("work.metadata_modules")
    )

    return work_metadata_modules

StatementMeta(, a683c6b6-3001-4232-958d-e4993d1e7ad9, 10, Finished, Available, Finished, False)

## Execute Standardization Process

Orchestrates the standardization workflow for the selected metadata entities.

In [ ]:
p_source_execution_step_ids = json.loads(
    p_source_execution_step_ids
)

p_metadata_entities = json.loads(
    p_metadata_entities
)

In [ ]:
if "metadata_objects" in p_metadata_entities:
    work_metadata_objects = process_metadata_objects(
        source_database=p_source_database,
        source_execution_step_id=p_source_execution_step_ids["metadata_objects"]
    )

if "metadata_columns" in p_metadata_entities:
    work_metadata_columns = process_metadata_columns(
        source_database=p_source_database,
        source_execution_step_id=p_source_execution_step_ids["metadata_columns"],
        objects_source_execution_step_id=p_source_execution_step_ids["metadata_objects"]
    )

if "metadata_constraints" in p_metadata_entities:
    work_metadata_constraints = process_metadata_constraints(
        source_database=p_source_database,
        source_execution_step_id=p_source_execution_step_ids["metadata_constraints"],
        objects_source_execution_step_id=p_source_execution_step_ids["metadata_objects"]
    )

if "metadata_constraint_columns" in p_metadata_entities:
    work_metadata_constraint_columns = process_metadata_constraint_columns(
        source_database=p_source_database,
        source_execution_step_id=p_source_execution_step_ids["metadata_constraint_columns"],
        constraints_source_execution_step_id=p_source_execution_step_ids["metadata_constraints"],
        columns_source_execution_step_id=p_source_execution_step_ids["metadata_columns"],
        objects_source_execution_step_id=p_source_execution_step_ids["metadata_objects"]
    )

if "metadata_relationships" in p_metadata_entities:
    work_metadata_relationships = process_metadata_relationships(
        source_database=p_source_database,
        source_execution_step_id=p_source_execution_step_ids["metadata_relationships"],
        objects_source_execution_step_id=p_source_execution_step_ids["metadata_objects"],
        columns_source_execution_step_id=p_source_execution_step_ids["metadata_columns"]
    )

if "metadata_modules" in p_metadata_entities:
    work_metadata_modules = process_metadata_modules(
        source_database=p_source_database,
        source_execution_step_id=p_source_execution_step_ids["metadata_modules"],
        objects_source_execution_step_id=p_source_execution_step_ids["metadata_objects"]
    )